# Human vs AI Forecasting

*Reproducible research notebook from [Flarient](https://flarient.com) — the space weather intelligence platform.*

**About this notebook:** This notebook is part of the [Flarient Research Notebooks](https://github.com/flarientglobal/flarient-notebooks) collection. It uses public data from NOAA SWPC, NASA, and the Flarient API.


## 1. Introduction

Can humans predict space weather better than AI? In this notebook, we'll explore the methodology for comparing human vs AI forecasting accuracy using the Flarient benchmark.


In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 6)


## 2. The Flarient Benchmark

The [Flarient Forecast Benchmark](https://github.com/flarientglobal/space-weather-forecast-benchmark) is an open dataset comparing human vs AI space weather forecasting accuracy.


In [ ]:
# Fetch benchmark leaderboard
benchmark_url = "https://raw.githubusercontent.com/flarientglobal/space-weather-forecast-benchmark/main/data/results/leaderboard.json"
try:
    response = requests.get(benchmark_url, timeout=15)
    if response.status_code == 200:
        leaderboard = response.json()
        print(f"Benchmark leaderboard: {len(leaderboard.get('leaderboard', []))} predictors")
        for entry in leaderboard.get('leaderboard', [])[:10]:
            print(f"  {entry['name']} ({entry['type']}): {entry.get('average_score', 0)} avg")
    else:
        print("Benchmark not yet available — visit the repo to submit forecasts")
except Exception as e:
    print(f"Benchmark fetch: {e}")


## 3. Scoring Methodology

The benchmark scores forecasts on multiple dimensions:


In [ ]:
# Scoring functions (from the benchmark)

def score_kp(forecast_kp, observed_kp):
    """Score Kp forecast: 100 - (MAE * 10)"""
    mae = abs(forecast_kp - observed_kp)
    return max(0, 100 - (mae * 10))

def score_flare(forecast_class, observed_class):
    """Score flare class: 100 if exact, 80 if 1 off, 50 if 2 off"""
    order = {'A': 0, 'B': 1, 'C': 2, 'M': 3, 'X': 4}
    diff = abs(order.get(forecast_class, 0) - order.get(observed_class, 0))
    return [100, 80, 50, 0][min(diff, 3)]

# Example scoring
print("Example Kp scoring:")
for f, o in [(5, 5), (5, 7), (3, 6), (8, 4)]:
    print(f"  Forecast {f}, Observed {o}: Score {score_kp(f, o)}")

print("\nExample Flare scoring:")
for f, o in [('M', 'M'), ('M', 'X'), ('C', 'M'), ('B', 'X')]:
    print(f"  Forecast {f}, Observed {o}: Score {score_flare(f, o)}")


## 4. Simulating Human vs AI Predictions

Let's simulate a comparison between human and AI predictors.


In [ ]:
# Simulate forecasts (in practice, these come from the benchmark)
np.random.seed(42)
n_forecasts = 50

# Human predictions: more variable, sometimes conservative
human_kp = np.random.normal(4.5, 1.5, n_forecasts).clip(0, 9)
human_flare = np.random.choice(['A', 'B', 'C', 'M', 'X'], n_forecasts, p=[0.1, 0.2, 0.5, 0.15, 0.05])

# AI predictions: more consistent, closer to observed
observed_kp = np.random.normal(4.0, 2.0, n_forecasts).clip(0, 9)
ai_kp = observed_kp + np.random.normal(0, 0.8, n_forecasts).clip(-3, 3)
ai_flare = np.random.choice(['A', 'B', 'C', 'M', 'X'], n_forecasts, p=[0.05, 0.15, 0.55, 0.2, 0.05])

# Calculate scores
human_scores = [score_kp(h, o) for h, o in zip(human_kp, observed_kp)]
ai_scores = [score_kp(a, o) for a, o in zip(ai_kp, observed_kp)]

print(f"Human avg Kp score: {np.mean(human_scores):.1f}")
print(f"AI avg Kp score: {np.mean(ai_scores):.1f}")


## 5. Visualising the Comparison

Let's create a comparison plot.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score distribution
axes[0].hist(human_scores, bins=20, alpha=0.6, color='#f59e0b', label='Human')
axes[0].hist(ai_scores, bins=20, alpha=0.6, color='#6366f1', label='AI')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Score Distribution: Human vs AI')
axes[0].legend()

# Cumulative accuracy
sorted_human = np.sort(human_scores)
sorted_ai = np.sort(ai_scores)
axes[1].plot(sorted_human, np.arange(1, len(sorted_human)+1)/len(sorted_human), 
             color='#f59e0b', linewidth=2, label='Human')
axes[1].plot(sorted_ai, np.arange(1, len(sorted_ai)+1)/len(sorted_ai), 
             color='#6366f1', linewidth=2, label='AI')
axes[1].set_xlabel('Score')
axes[1].set_ylabel('Cumulative Proportion')
axes[1].set_title('Cumulative Score: Human vs AI')
axes[1].legend()

plt.tight_layout()
plt.savefig('human_vs_ai.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Submit Your Own Forecast

You can participate in the benchmark! Visit the [Flarient Forecast Benchmark](https://github.com/flarientglobal/space-weather-forecast-benchmark) to submit your predictions and see how you compare.

For live human vs AI comparisons on Flarient, visit [flarient.com/human-vs-ai](https://flarient.com/human-vs-ai).


## 7. Conclusion

This notebook showed:
1. How the Flarient benchmark scores forecasts
2. A simulated comparison of human vs AI predictions
3. How to participate in the open benchmark

The key insight: transparency in forecasting matters. The [Event Ledger](https://github.com/flarientglobal/flarient-event-ledger) provides verifiable proof of when predictions were made.


---

## About Flarient

[Flarient](https://flarient.com) is a space weather intelligence platform providing real-time data, forecasts, and community-driven observations. Visit [flarient.com](https://flarient.com) for live space weather conditions, aurora forecasts, and more.

## License

MIT — This notebook is open source. [View on GitHub](https://github.com/flarientglobal/flarient-notebooks).
